# Notebook 27 — D=7 Barlow-Twins Embedding: Trajectories + 2-D Views (standalone)
**Project:** ENSO-BSISO SSL — MJO extension  
**Author:** Jiayi (jh9141@nyu.edu)

Lightweight, **no-retrain** viewer for the **D=7** Barlow-Twins embedding from nb26. Loads the saved
`embeddings_z7.npy` (no model / torch / GPU needed) and reproduces, for D=7:
- PCA + t-SNE 2-D scatter colored by RMM phase / ENSO / amplitude (nb26 Cell 9), and
- **event trajectories** — consecutive days of the strongest MJO events as connected paths through PCA-2D (nb26 Cell 10).

**Reuse vs retrain:** *reuse* — the D=7 embedding was already saved by the earlier nb26 D=7 run to `MJO/barlow/embeddings_z7.npy`. (The later D=3 runs wrote to `MJO/barlow/D3/`, so the base file is still D=7.) Only re-run nb26 with `PROJ_DIM=7` if that file is missing or not 7-dimensional (Cell 1 checks).

**Inputs:** `MJO/barlow/embeddings_z7.npy` (N, 7); `MJO/data/processed/labels_aligned_mjo_bp20_90.csv`.  
**Outputs:** `MJO/barlow/D7_embedding_2d.png`, `MJO/barlow/D7_trajectories_pca.png`.

---

## Cell 1 — Load the saved D=7 embedding + labels (aligned)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR       = f'{PROJECT_DIR}/MJO'
PROCESSED_DIR = f'{MJO_DIR}/data/processed'
OUT_DIR       = f'{MJO_DIR}/barlow'

# saved D=7 embedding (base barlow/ folder; D=3 runs are under barlow/D3/)
EMB_CANDIDATES = [f'{OUT_DIR}/embeddings_z7.npy', f'{OUT_DIR}/D7/embeddings_z7.npy']
emb_path = next((p for p in EMB_CANDIDATES if os.path.exists(p)), None)
assert emb_path is not None, ('No saved embedding found. Re-run nb26 with PROJ_DIM=7 to create '
                              'MJO/barlow/embeddings_z7.npy.')
z7 = np.load(emb_path)
print(f'Loaded {emb_path}  shape={z7.shape}')
if z7.shape[1] != 7:
    print(f'WARNING: embedding dim is {z7.shape[1]}, not 7. This file is from a non-D=7 run; '
          're-run nb26 with PROJ_DIM=7 if you want the D=7 trajectories.')

# labels, sorted by date to match the order nb26 saved the embedding in (nb26 Cell 2 sorts)
labels = pd.read_csv(f'{PROCESSED_DIR}/labels_aligned_mjo_bp20_90.csv', parse_dates=['date'])
labels['date'] = labels['date'].dt.normalize()
labels = labels.iloc[np.argsort(labels['date'].values)].reset_index(drop=True)
assert len(labels) == len(z7), f'labels {len(labels)} != embedding {len(z7)} (different preprocessing run?)'

N         = len(z7)
dates_all = pd.DatetimeIndex(labels['date'].values)
phase_all = labels['phase'].values.astype(int)
amp_all   = labels['amplitude'].values
enso_all  = labels['enso_category'].values
weak_all  = labels['weak_mjo'].values.astype(bool)
active    = (~weak_all) & (amp_all >= 1.0)
print(f'N={N}  active(amp>=1)={int(active.sum())}  ({dates_all.min().date()}..{dates_all.max().date()})')

## Cell 2 — PCA + t-SNE 2-D scatter (by phase / ENSO / amplitude)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

idxa = np.where(active)[0]
rng  = np.random.default_rng(0)
sub  = rng.choice(idxa, size=min(5000, len(idxa)), replace=False)
Zs   = z7[sub]
ph   = phase_all[sub]; en = enso_all[sub]; am = amp_all[sub]

pca2  = PCA(2).fit_transform(Zs - Zs.mean(0))
tsne2 = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto',
             random_state=0).fit_transform(Zs)

phase_colors = plt.cm.hsv(np.linspace(0, 1, 9))   # cyclic: a phase LOOP would order around the wheel
enso_pal = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}

def scatter_by(ax, XY, kind):
    if kind == 'phase':
        for p in range(1, 9):
            m = ph == p
            ax.scatter(XY[m, 0], XY[m, 1], s=6, alpha=0.6, color=phase_colors[p-1], label=f'P{p}')
        ax.set_title('by RMM phase'); ax.legend(fontsize=6, ncol=2, markerscale=2)
    elif kind == 'enso':
        for c in ['El Nino', 'Neutral', 'La Nina']:
            m = en == c
            ax.scatter(XY[m, 0], XY[m, 1], s=6, alpha=0.5, color=enso_pal[c], label=c)
        ax.set_title('by ENSO'); ax.legend(fontsize=7, markerscale=2)
    else:
        sc = ax.scatter(XY[:, 0], XY[:, 1], s=6, alpha=0.6, c=am, cmap='viridis')
        ax.set_title('by RMM amplitude'); plt.colorbar(sc, ax=ax, fraction=0.046)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for j, kind in enumerate(['phase', 'enso', 'amplitude']):
    scatter_by(axes[0, j], pca2,  kind)
    scatter_by(axes[1, j], tsne2, kind)
axes[0, 0].set_ylabel('PCA-2D', fontsize=12)
axes[1, 0].set_ylabel('t-SNE-2D', fontsize=12)
plt.suptitle(f'D=7 Barlow-Twins embedding -> 2-D  (n={len(sub)} active days)',
             fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.98])
p = f'{OUT_DIR}/D7_embedding_2d.png'
plt.savefig(p, dpi=130, bbox_inches='tight'); plt.show(); print('Saved', p)

## Cell 3 — Event Trajectories (connected paths in PCA-2D)

Connect consecutive days of the 3 strongest MJO events as paths through the D=7 embedding (PCA-2D), over a faint background of all active days. PCA only (linear → continuity is faithful); t-SNE is avoided for paths (non-metric → misleading lines). ★ start, ■ end, markers colored by RMM phase. A clean MJO event would trace a phase-ordered loop.

In [ ]:
from sklearn.decomposition import PCA

mu = z7[idxa].mean(0)
P2 = PCA(2).fit(z7[idxa] - mu)
bg = P2.transform(z7[idxa] - mu)

# contiguous active runs (consecutive calendar days, amplitude >= 1) of length >= 25
runs = []; i = 0
while i < N:
    if active[i]:
        j = i
        while j + 1 < N and active[j + 1] and int((dates_all[j + 1] - dates_all[j]).days) == 1:
            j += 1
        if j - i + 1 >= 25:
            runs.append((i, j))
        i = j + 1
    else:
        i += 1
runs = sorted(runs, key=lambda r: amp_all[r[0]:r[1] + 1].mean(), reverse=True)[:3]
print(f'Plotting {len(runs)} strongest MJO events (>=25 consecutive active days).')

phase_colors = plt.cm.hsv(np.linspace(0, 1, 9))
fig, ax = plt.subplots(figsize=(10, 9))
ax.scatter(bg[:, 0], bg[:, 1], s=4, c='lightgray', alpha=0.35, zorder=1)
for (a, b) in runs:
    seg = P2.transform(z7[a:b + 1] - mu)
    ph  = phase_all[a:b + 1]
    ax.plot(seg[:, 0], seg[:, 1], '-', color='k', lw=1.0, alpha=0.6, zorder=2)
    ax.scatter(seg[:, 0], seg[:, 1], c=[phase_colors[p - 1] for p in ph], s=40,
               edgecolor='k', linewidth=0.3, zorder=3)
    ax.scatter(*seg[0],  marker='*', s=320, c='lime', edgecolor='k', zorder=4)
    ax.scatter(*seg[-1], marker='s', s=110, c='red',  edgecolor='k', zorder=4)
    ax.annotate(f'{dates_all[a].date()} -> {dates_all[b].date()}', seg[0],
                fontsize=8, xytext=(6, 6), textcoords='offset points')
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='o', ls='', mfc=phase_colors[p - 1], mec='k', label=f'P{p}') for p in range(1, 9)]
ax.legend(handles=handles, fontsize=8, ncol=2, title='RMM phase', loc='best')
ax.set_title('MJO event trajectories in D=7 embedding (PCA-2D)\n'
             '* start, square end; markers colored by RMM phase; grey = all active days', fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
p = f'{OUT_DIR}/D7_trajectories_pca.png'
plt.savefig(p, dpi=140, bbox_inches='tight'); plt.show(); print('Saved', p)
print('A clean MJO event would trace a LOOP with phases 1->8 ordered around it.')

---
## Done!

Two figures in `MJO/barlow/`: `D7_embedding_2d.png` (PCA+t-SNE scatter) and `D7_trajectories_pca.png` (event paths).

No retraining was needed — this reuses the saved D=7 `embeddings_z7.npy`. If you ever want the *exact* numbers/figures regenerated from the model, re-run nb26 with `PROJ_DIM=7` instead.

---
*DDCS Project | jh9141@nyu.edu*